# Correct Photometry for dust extinction

Correct broad-band photometry for Host and MW extinction (if this has not been already done.. 99% of the time the photometry is published before dust corrections).

Assumptions: R_V always equal 3.1


In [1]:
import pipeline_config as pconf
from pipeline_config import bootstrap_runtime

rt = pconf.bootstrap_runtime(photometry_stage="raw", output_stage="dust_corrected")
OUTPUT_DIR = rt.output_dir
OUTPUT_PATH = rt.output_path
DATALC_PATH = rt.datalc_path
DATASPEC_PATH = rt.dataspec_path
DATAINFO_PATH = rt.datainfo_path
FILTER_PATH = rt.filter_path
FILTER_LEAF = rt.filter_leaf
FILTERS_PARENT = rt.filters_parent
color_dict, mark_dict, exclude_filt = rt.color_dict, rt.mark_dict, rt.exclude_filt
FINAL_SPECTRA_DIR = rt.final_spectra_dir
GP_MODE = rt.gp_mode
import what_the_flux as wtf


In [ ]:
# exclude_filt comes from bootstrap_runtime / filter_plot_config.json
# Optional per-run override: exclude_filt = exclude_filt + ['SomeBand']


In [10]:
import numpy as np
import pandas as pd
import os

In [3]:
#trying without infrared, that's why commented out old ones
#AB_filt = ['sdss_g','sdss_r','sdss_i','sdss_z', "Bessell_B", "Bessell_V", "Bessell_U", "Bessell_I",
#           "H", "J", "Ks", "Y"]
AB_filt = ['sdss_g','sdss_r','sdss_i','sdss_z', "Bessell_B", "Bessell_V", "Bessell_U", "Bessell_I"]
Vega_filt = ['swift_UVW2']

CSP_SNe = ['SN2004fe', 'SN2005bf', 'SN2006V', 'SN2007C', 'SN2007Y',
           'SN2009bb',  'SN2008aq', 'SN2006T', 'SN2004gq', 'SN2004gt',
           'SN2004gv','SN2006ep', 'SN2008fq', 'SN2006aa']

#exclude_filt = ['K']
#exclude_filt = ['H', 'J', 'K', 'Ks','KS', 'Y']

info_objects = pd.read_csv(DATAINFO_PATH+'info.dat', comment='#', delimiter=' ')

In [4]:
# AB_filt = ['Swope_i', 'FourStar_H', 'FourStar_J', 'Sinistro_w', 'FourStar_Ks',
#        'Sinistro_g', 'Sinistro_r', 'EFOSC2_V', 'Sinistro_i', 'Swope_V',
#        'FourStar_J1', 'Swope_B', 'Swope_r', 'EFOSC2_U', 'Swope_g',
#        'Sinistro_z', 'IMACS_r', 'RetroCam_Y', 'RetroCam_H', 'RetroCam_J',
#        'LRIS_I', 'LRIS_g', 'SOFI_H', 'SOFI_Ks']
#exclude_filt = ['H', 'J', 'K', 'Ks','KS', 'Y', 'FourStar_H', 'FourStar_J', 'FourStar_Ks', 'FourStar_J1',
#                'RetroCam_Y', 'RetroCam_H', 'RetroCam_J', 'SOFI_H', 'SOFI_Ks']

# AB_filt = ['RetroCam_J',
#  'Swope_i',
#  'ACS_WFC_F625W',
#  'UVOT_U',
#  'DECam_i',
#  'RetroCam_H',
#  'DECam_z',
#  'DECam_Y',
#  'RetroCam_Y',
#  'FourStar_H',
#  'EFOSC2_r',
#  'IMACS_V2',
#  'Sinistro_z',
#  'EFOSC2_g',
#  'FourStar_J',
#  'IMACS_V1',
#  'ACS_WFC_F475W',
#  'IMACS_r',
#  'skymapper_g',
#  'EFOSC2_U',
#  'LRIS_I',
#  'Sinistro_i',
#  'skymapper_r',
#  'EFOSC2_V',
#  'SOFI_H',
#  'Sinistro_g',
#  'LRIS_g',
#  'skymapper_i',
#  'WFC3_UVIS1_F336W',
#  'Sinistro_r',
#  'WFC3_IR_F160W',
#  'FourStar_Ks',
#  'Sinistro_V',
#  'Sinistro_w',
#  'FourStar_J1',
#  'SOFI_Ks',
#  'EFOSC2_i',
#  'DECam_u',
#  'Swope_V',
#  'Swope_B',
#  'ACS_WFC_F850W',
#  'DECam_r',
#  'WFC3_IR_F110W',
#  'Swope_g',
#  'Swope_r',
#  'ACS_WFC_F775W',
#  'DECam_g']

AB_filt = ['Swope_i', 'FourStar_H', 'VISTA_Ks', 'FourStar_J', 'DECam_i',
       'DECam_z', 'VISTA_J', 'VISTA_Y', 'FourStar_Ks', 'FLAMINGOS-2_Ks',
       'UVOT_M2', 'UVOT_W1', 'UVOT_U', 'UVOT_W2', 'HSC_z', 'GFC_i',
       'GFC_y', 'GFC_z', 'Skymapper_i', 'Sinistro_g', 'Sinistro_r',
       'Skymapper_r', 'Skymapper_g', 'SIRIUS_H', 'SIRIUS_J', 'SIRIUS_Ks',
       'EFOSC2_V', 'T80Cam_g', 'GROND_H', 'GROND_J', 'GROND_K', 'GROND_g',
       'GROND_i', 'GROND_r', 'GROND_z', 'Sinistro_i', 'DECam_Y',
       'DECam_r', 'DECam_g', 'DECam_u', 'Swope_V', 'FLAMINGOS-2_H',
       'FourStar_J1', 'Swope_B', 'Swope_r', 'Swope_g', 'EFOSC2_U',
       'Sinistro_V', 'Sinistro_z', 'GFC_r', 'IMACS_i', 'IMACS_r',
       'FLAMINGOS-2_J', 'UVOT_B', 'GMOS_g', 'GMOS_i', 'GMOS_r', 'GMOS_z',
       'FORS2_R', 'VIMOS_z', 'FORS2_I', 'FORS2_B', 'FORS2_V', 'LRIS_I',
       'ANDICAM_K', 'SOFI_H', 'SOFI_Ks', 'MOIRCS_Ks', 'HAWKI_Ks']

# exclude_filt from filter_plot_config.json via bootstrap


In [5]:
info_objects

,Name,z,RA,Dec,FullType,Type,Rich_Type,EBV_MW,EBV_host
0,iPTF13bvn,0.004490,15:00:00.15,+01:52:53.17,Ib,Ib,Ib,0.0510,0.170
1,SN2011bm,0.022000,12:56:53.89,+22:22:28.2,Ic,Ic,Ic,0.0340,0.032
2,SN1993J,-0.000113,09:55:24.7747,+69:01:13.702,IIb,IIb,IIb,0.0800,0.100
3,AT2017gfo,0.009840,13:09:48.082,-23:22:53.28,KN,KN,KN,0.1053,0.000


In [6]:

class SNPhotometryClass():
    """Class with photometry for each object:
            - load the photometry from the DATA folder
            - get the phootmetry in each filter
            - plot the raw photometry 
            - fit the photometry using GP
    """
    
    def __init__(self, lc_path, snname, verbose=False):
        """
        """
        self.lc_data_path = lc_path+''
        self.snname = snname   
        self.set_data_directory(verbose)

    def set_data_directory(self, verbose):
        """
        Set a new data directory path.
        Enables the data directory to be changed by the user.
        """
        SNphotometry_PATH = os.path.join(self.lc_data_path, '%s.dat'%self.snname)
        
        try:
            if verbose: print('Looking for Photometry for %s in%s'%(self.snname, SNphotometry_PATH))
            if os.path.isfile(SNphotometry_PATH):
                if verbose: print ('Got it!')
                self.sn_rawphot_file = SNphotometry_PATH
                pass
            else:
                if not os.path.isdir(self.lc_data_path):
                    print ('I cant find the directory with photometry. Check %s'%self.lc_data_path)
                    pass
                else: 
                    print ('I cant find the file with photometry. Check %s'%SNphotometry_PATH)
                    pass
    
        except Exception as e:
            print (e)

    def load(self, verbose = False):
        """
        Loads a single photometry file.
        with ('MJD', 'flux', 'flux_err', 'filter')
        
        Parameters
        - verbose
        ----------
        Returns
        - photometry in all filters
        -------
        """
        if verbose: print('Loading %s'%self.sn_rawphot_file)
        try:
            lc_file = pd.read_csv(self.sn_rawphot_file,
                                    dtype=None,encoding="utf-8")
            mask_filt = np.array([f not in exclude_filt for f in lc_file['band']])
            lc_no_badfilters = lc_file[mask_filt]
            mask_filt = np.array([~np.isnan(f) for f in lc_no_badfilters['Flux']])
            self.phot = lc_no_badfilters[mask_filt]
            self.avail_filters = np.unique(self.phot['band'])
            if verbose: print ('Photometry loaded')

        except Exception as e:
            print (e)
            print ('Are you sure you gave me the right format? Check documentation in case.')

    def get_availfilter(self, verbose = False):
        """
        get available filter for this SN
        """
        #if photometry is not already loaded, load it!
        if (not hasattr(self, "phot"))|(not hasattr(self, "avail_filters")):
            self.load()
        return self.avail_filters
        
    def get_singlefilter(self, single_filter, verbose = False):
        """
        Loads from photometry file just 1 filter photometry.
        with ('MJD', 'flux', 'flux_err', 'filter')
        
        Parameters
        - verbose
        ----------
        Returns
        - photometry in all filters
        -------
        """
        #if photometry is not already loaded, load it!
        if not hasattr(self, "phot"):
            self.load()

        if not (isinstance(single_filter, str)):
            print ('Single filter string please')
            return None
        
        if single_filter not in self.avail_filters:
            if verbose: print ('Looks like the filter you are looking for is not available')
            return None
        
        filt_index = self.phot['band']==single_filter
        return self.phot[filt_index]
        
    def corr_dust_singlefilter(self, filter_name):
        if not hasattr(self, "corr_factors"):
            self.corr_factors = {}
        corr_factors_dict = self.corr_factors            

        self.get_dust()
        self.get_redshift()
        RV = RV_dict[self.SNType]

        if 'swift' in filter_name:
            w,t = w,t=wtf.loadFilter(FILTER_PATH+'/Swift/%s.dat'%filter_name)
        elif self.snname in CSP_SNe:
            w,t = w,t=wtf.loadFilter(FILTER_PATH+'/Site3_CSP/%s.txt'%filter_name)
        else:
            w,t = w,t=wtf.loadFilter(FILTER_PATH+'/GeneralFilters/%s.dat'%filter_name)
            
        w_SNframe = w/(1.+self.redshift)
        
        #if filter_name in Vega_filt: band = wtf.Band_Vega(w_SNframe,t)
        #elif filter_name in AB_filt: band = wtf.Band_AB(w_SNframe,t)
        if filter_name in Vega_filt:
            band = wtf.Band_Vega(w_SNframe, t)
        elif filter_name in AB_filt:
            band = wtf.Band_AB(w_SNframe, t)
        else:
            raise ValueError(f"Filter '{filter_name}' not found in Vega_filt or AB_filt. Please check your filter lists.")
        ext_corr_Host = (1./band.extinction(self.Hostebv, 'CCM', r_v = RV)).value
        print (filter_name, 'RV',RV, 'Host dust correction %.3f'%ext_corr_Host)

        if filter_name in Vega_filt: band = wtf.Band_Vega(w,t)
        elif filter_name in AB_filt: band = wtf.Band_AB(w,t)
        ext_corr_MW = (1./band.extinction(self.MWebv, 'CCM')).value

        corr_factors_dict[filter_name] = ext_corr_Host*ext_corr_MW
        self.corr_factors = corr_factors_dict           

    
    def get_dust(self):
        if self.snname not in info_objects.Name.values:
            raise Exception('This SN is not in the info.dat file')
        else:
            info_singleobj = info_objects[info_objects.Name==self.snname]
            self.MWebv = info_singleobj['EBV_MW'].values[0]
            Host_ebv = info_singleobj['EBV_host'].values[0]
            self.Hostebv = Host_ebv
            self.SNType = (info_singleobj['Type'].values[0])
        
    def get_redshift(self):
        info_singleobj = info_objects[info_objects.Name==self.snname]
        self.redshift = info_singleobj['z'].values[0]

    def correct_final_LC(self, name_file = None):
        for ff in self.get_availfilter():
            self.corr_dust_singlefilter(ff)
        
        lc_file = pd.DataFrame(self.phot)
        corr_dust_array = [self.corr_factors[f] for f in lc_file['band']]
        lc_file['Flux_corr'] = corr_dust_array*lc_file['Flux'].values
        lc_file['Flux_corr_err'] = corr_dust_array*lc_file['Flux_err'].values

        return lc_file

## Dust Correction

In [7]:
# If you want to do something more sophisticated and set a different R_V for different SN types set this:
# RV_dict = {'Ic':4.3 , 'Ic-BL':4.3, 'Ib':2.6, 'IIb':1.1, 'II':3.1, 'IIn':3.1}
RV_dict = {'KN': 3.1}

In [8]:
snname = 'AT2017gfo'

In [11]:
sn_phot = SNPhotometryClass(lc_path=DATALC_PATH,snname=snname, verbose=True)
sn_phot.load()

Looking for Photometry for AT2017gfo in/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Inputs/Photometry/1_LCs_flux_raw/AT2017gfo.dat
Got it!


In [12]:
sn_phot.get_availfilter()
df = sn_phot.correct_final_LC()

df_to_print = df[['MJD', 'band', 'Flux_corr', 'Flux_corr_err', 'FilterSet', 'Instr']]
df_to_print.to_csv(OUTPUT_PATH+'/%s.dat'%snname, na_rep='nan', index=False, 
                           header=['#MJD','band','Flux','Flux_err','FilterSet', 'Instr'])

ANDICAM_K RV 3.1 Host dust correction 1.000
DECam_Y RV 3.1 Host dust correction 1.000
DECam_g RV 3.1 Host dust correction 1.000
DECam_i RV 3.1 Host dust correction 1.000
DECam_r RV 3.1 Host dust correction 1.000
DECam_u RV 3.1 Host dust correction 1.000
DECam_z RV 3.1 Host dust correction 1.000
EFOSC2_U RV 3.1 Host dust correction 1.000
FLAMINGOS-2_H RV 3.1 Host dust correction 1.000
FLAMINGOS-2_J RV 3.1 Host dust correction 1.000
FLAMINGOS-2_Ks RV 3.1 Host dust correction 1.000
FORS2_B RV 3.1 Host dust correction 1.000
FORS2_I RV 3.1 Host dust correction 1.000
FORS2_R RV 3.1 Host dust correction 1.000
FORS2_V RV 3.1 Host dust correction 1.000
FourStar_H RV 3.1 Host dust correction 1.000
FourStar_J RV 3.1 Host dust correction 1.000
FourStar_J1 RV 3.1 Host dust correction 1.000
FourStar_Ks RV 3.1 Host dust correction 1.000
GFC_i RV 3.1 Host dust correction 1.000
GFC_r RV 3.1 Host dust correction 1.000
GFC_y RV 3.1 Host dust correction 1.000
GFC_z RV 3.1 Host dust correction 1.000
GMOS_g